# Personal AI Tax Adviser

### Loading all the pdfs for rag

In [1]:
!pip install -qU langchain-community pypdf

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = [
    "/content/drive/MyDrive/pdfs for rag/CBDT_e-Filing_ITR 4_Validation Rules_AY 2026-27.pdf",
    "/content/drive/MyDrive/pdfs for rag/Common ITR Filing FAQs AY 2024-25.pdf",
    "/content/drive/MyDrive/pdfs for rag/Financial Education Booklet - English.pdf",
    "/content/drive/MyDrive/pdfs for rag/ITR-7_FAQ_AY_2024-25.pdf",
    "/content/drive/MyDrive/pdfs for rag/New vs. Old Regime FAQs approved final.pdf",
]

all_documents = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from {path.split('/')[-1]}")
    all_documents.extend(documents)

/tmp/ipykernel_35066/292998271.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 24 pages from CBDT_e-Filing_ITR 4_Validation Rules_AY 2026-27.pdf
Loaded 5 pages from Common ITR Filing FAQs AY 2024-25.pdf
Loaded 73 pages from Financial Education Booklet - English.pdf
Loaded 17 pages from ITR-7_FAQ_AY_2024-25.pdf
Loaded 5 pages from New vs. Old Regime FAQs approved final.pdf


### Split, Embed, and Store

In [3]:
!pip install -qU langchain langchain-core langchain-community langchain-text-splitters \
    langchain-huggingface langchain-chroma pypdf nltk chromadb

In [6]:
#!pip install -U --quiet opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp-proto-common opentelemetry-proto chromadb

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(all_documents)
print(f"Total chunks: {len(chunks)}")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma(
    collection_name="finance_docs",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)
vector_store.add_documents(documents=chunks)
print("Knowledge base ready!")

Total chunks: 346


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base ready!


###Converting to a Tool

In [5]:
from langchain.tools import tool

@tool
def search_tax_docs(question: str) -> str:
    """Search official income tax documents and SEBI financial education guides for Section 80C/80D deductions, old vs new tax regime comparison, ITR filing guidance, and tax saving investments"""
    results = vector_store.similarity_search(question, k=3)
    if not results:
        return "No relevant information found in the knowledge base."
    context = ""
    for doc in results:
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        context += f"Source: {source} (Page {page + 1})\n"
        context += f"Content: {doc.page_content}\n\n"
    return context

###The Full AI Advisor

In [6]:
!pip install -qU langchain-groq

In [7]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')

model = init_chat_model(
    "groq:llama-3.3-70b-versatile",
    api_key=api_key,
)

In [8]:
system_prompt = """You are a Personal Tax Advisor for Indian citizens.

You have access to these tools:
- search_tax_docs: Search official Indian government documents for tax saving
  options (80C, 80D), old vs new tax regime, ITR filing, and investment guidance

Help the user by looking up the relevant data using your tools and giving clear,
specific answers with actual numbers and rates. Always mention the source of
your information. All monetary values should be in Indian Rupees (₹) unless
specified otherwise.
"""

agent = create_agent(
    model=model,
    tools=[search_tax_docs],
    system_prompt=system_prompt,
)

In [9]:
import requests

response = agent.invoke({
    "messages": [{"role": "user", "content": "What's the difference between the old and new tax regime?"}]
})
print("What's the difference between the old and new tax regime?")
print(response["messages"][-1].content)

response = agent.invoke({
    "messages": [{"role": "user", "content": "Can I claim deduction for health insurance premium under Section 80D? What's the limit?"}]
})
print("Can I claim deduction for health insurance premium under Section 80D? What's the limit?")
print(response["messages"][-1].content)


What's the difference between the old and new tax regime?
The main difference between the old and new tax regime is that the new tax regime offers lower tax rates, but it does not provide exemptions and deductions that are available in the old tax regime. According to the Income Tax Portal, taxpayers can estimate and compare their tax liability under both regimes using the Income and Tax Calculator. Ultimately, the choice between the two regimes depends on individual circumstances and requirements. (Source: Income Tax Portal)
Can I claim deduction for health insurance premium under Section 80D? What's the limit?
According to the Income Tax Department, Government of India, you can claim a deduction for health insurance premium under Section 80D. The limit for this deduction is ₹25,000 for individuals and ₹50,000 for senior citizens. However, an additional deduction of ₹25,000 is allowed for insurance of parents. If the parents are senior citizens, the additional deduction allowed is ₹50

## Building the UI

In [11]:
!pip install -qU gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00


In [13]:
import gradio as gr

def extract_text(message):
    """Extract text from an LLM message, handling both string and list content formats."""
    return message.text

def tax_advisor(question):
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    last_message = response["messages"][-1]
    return extract_text(last_message)

demo = gr.Interface(
    fn=tax_advisor,
    inputs=gr.Textbox(lines=2, placeholder="Ask a tax question...", label="Question"),
    outputs=gr.Textbox(lines=10, label="Answer"),
    title="Personal Tax AI Advisor",
    description="Ask about Section 80C/80D deductions, old vs new tax regime, ITR filing, and tax saving investments.",
)
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://584bca56c4ed21aa67.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://584bca56c4ed21aa67.gradio.live


In [12]:
!pip install -q -U ragas langchain-google-genai langchain-community

In [13]:
!pip install -q langchain-google-genai

In [14]:
!pip uninstall -y ragas langchain-community
!pip install -q ragas==0.3.9 "langchain-community<0.4" langchain-google-genai

Found existing installation: ragas 0.4.3
Uninstalling ragas-0.4.3:
  Successfully uninstalled ragas-0.4.3
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00


In [15]:
!pip show ragas langchain-community | grep -iE "^(Name|Version)"

Name: ragas
Version: 0.3.9
Name: langchain-community
Version: 0.3.31


In [16]:
from ragas.llms import LangchainLLMWrapper
print("ragas import OK")

ragas import OK


In [17]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall
from ragas import evaluate, EvaluationDataset

# 1. Set API key (pull from Colab secrets, same pattern as your gemini_api_key secret)
os.environ["GOOGLE_API_KEY"] = userdata.get('gemini_api_key')

# 2. Setup LLM
evaluator_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=os.environ["GOOGLE_API_KEY"]
    )
)
print("LLM ready:", evaluator_llm)

# 3. Define your data
eval_data = [
    {
        "user_input": "What is the maximum deduction available under Section 80C?",
        "retrieved_contexts": ["Section 80C of the Income Tax Act allows a maximum deduction of ₹1,50,000 per financial year through eligible investments such as PPF, ELSS, life insurance premiums, EPF contributions, and tuition fees."],
        "response": "The maximum deduction under Section 80C is ₹1,50,000 per financial year.",
        "reference": "Section 80C allows a maximum deduction of ₹1,50,000 per financial year.",
    },
    {
        "user_input": "Can I claim deduction for health insurance premium under Section 80D?",
        "retrieved_contexts": ["Section 80D allows a deduction of up to ₹25,000 per year for health insurance premiums paid for self, spouse, and dependent children. An additional deduction of ₹25,000 (₹50,000 for senior citizen parents) is available for premiums paid for parents."],
        "response": "Yes, under Section 80D you can claim up to ₹25,000 for self and family, plus an additional ₹25,000 (₹50,000 if parents are senior citizens) for parents' premiums.",
        "reference": "Section 80D allows a deduction of up to ₹25,000 for self, spouse, and children, and an additional ₹25,000 (₹50,000 for senior citizen parents) for parents' health insurance premiums.",
    },
]

# 4. Create dataset
eval_dataset = EvaluationDataset.from_list(eval_data)
print(f"Dataset size: {len(eval_dataset.samples)}")

# 5. Define metrics
faithfulness = Faithfulness(llm=evaluator_llm)
context_precision = ContextPrecision(llm=evaluator_llm)
context_recall = ContextRecall(llm=evaluator_llm)

# 6. Evaluate
results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, context_precision, context_recall],
    llm=evaluator_llm,
)
print(results)

/tmp/ipykernel_35066/3971752265.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(


LLM ready: LangchainLLMWrapper(langchain_llm=ChatGoogleGenerativeAI(...))
Dataset size: 2


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'context_precision': 1.0000, 'context_recall': 1.0000}
